# Postprocessing

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import h5py
import numpy as np
import pandas as pd
from matplotlib.patches import Circle
import os
import myfunctions
import h5py
import TM26_unet as unet


GPU = True
MULTIGPU = True

DELETE_OLD_FILES = False
DELETE_OLD_FILES = False

SMALL_SIZE = 18
MEDIUM_SIZE = 22
BIGGER_SIZE = 24
GPU = True

diffLimDir = "dataInputs/"
testDataDir = "output_prep/"
outputDir = "output_post/"
inputDir = "output_infer/"

# datasets_name = ["lightfield", "WFSxy", "WFSxy_DM", "WFSxy_DM_intensity", "WFSxy_intensity"]
datasets_name = ["WFSxy_intensity"]
dataset_name_readable = ["lightfield", "WFSxy", "WFSxy and DM commands", "WFSxy, SHWFS intensity map and DM commands", "WFSxy and SHWFS intensity map"]


D = 17.5e-3  # [m]
wl = 632.8e-9  # [m]
pad = 2
dx = 3.928571e-06

SMALL_SIZE = 18
MEDIUM_SIZE = 22
BIGGER_SIZE = 24
GPU = True



dataset_numbers = len(datasets_name)

### Inference data

In [ ]:
for dataset_number in range(dataset_numbers):
    if datasets_name[dataset_number] == "lightfield":
        with h5py.File(inputDir+datasets_name[dataset_number]+"/predictions.h5", 'r') as temp:
            inferences = temp[list(temp.keys())[0]][()]
        with h5py.File("dataInputs/ipsf_difflim.h5", 'r') as temp:
            ipsf_difflim = temp[list(temp.keys())[0]][()]   
        truths = np.zeros_like(inferences)
        with h5py.File(testDataDir+"ao_ipsf_test_1.h5", 'r') as temp:
            key = list(temp.keys())
            tempdata = temp[key[0]][()][:]
            for i in range(tempdata.shape[0]):
                truths[i,:,:] = myfunctions.pixel_adder(tempdata[i],final_shape=[inferences.shape[1], inferences.shape[2]])
        
    else:
        with h5py.File(inputDir+datasets_name[dataset_number]+"/predictions.h5", 'r') as temp:
            inferences = temp[list(temp.keys())[0]][()]
        with h5py.File("dataInputs/ipsf_difflim.h5", 'r') as temp:
            ipsf_difflim = temp[list(temp.keys())[0]][()]
        with h5py.File(testDataDir+"ao_ipsf_test_1.h5", 'r') as temp:
            key = list(temp.keys())
            truths = temp[key[0]][()][:]
    padded_ipsf_difflim = myfunctions.pad_center(ipsf_difflim, pad)        
    padded_truth = myfunctions.pad_center(truths, pad)
    padded_inferences = myfunctions.pad_center(inferences,pad)

    N = ipsf_difflim.shape[1]*pad
    fc = D / wl
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=dx))
    freq_n = np.abs(freq) / fc
    NU_X, NU_Y = np.meshgrid(freq_n, freq_n)
    nu_n_grid = np.sqrt(NU_X**2 + NU_Y**2)
    # FREQ_MASK = np.abs(freq) < fc
    FREQ_MASK = nu_n_grid < 1

    OTF_ipsf_difflim = np.abs(myfunctions.otf_from_psf(padded_ipsf_difflim))
    OTF_tel = myfunctions.telescope_otf(nu_n_grid)
    OTF_aliasing = OTF_ipsf_difflim - OTF_tel
    OTF_truths = (np.abs(myfunctions.otf_from_psf(padded_truth))[...,:,:] - OTF_aliasing)*FREQ_MASK
    OTF_inferences = (np.abs(myfunctions.otf_from_psf(padded_inferences))[...,:,:] - OTF_aliasing)*FREQ_MASK

    inference_Strehls = np.sum(OTF_inferences, axis=(-2, -1)) / np.sum(OTF_tel*FREQ_MASK)
    truths_Strehls = np.sum(OTF_truths, axis=(-2, -1)) / np.sum(OTF_tel*FREQ_MASK)
    Strehls_difference = truths_Strehls - inference_Strehls
    Strehls_rel_fidd = Strehls_difference/truths_Strehls


therandom = np.random.randint(0, inferences.shape[0])
print(therandom)
plt.figure(figsize=(10, 8))
plt.plot(freq, OTF_truths[therandom,N//2, :], label="Measured OTF")
plt.plot(freq, OTF_inferences[therandom,N//2, :]*FREQ_MASK[N//2,:], label="Inference OTF")
plt.plot(freq, OTF_tel[N//2, :], "k", label="Theoretical diffraction-limited OTF")
plt.plot(freq, OTF_ipsf_difflim[N//2, :], label="Diffraction-limited OTF\nfrom simulated PSF with aliasing")
plt.plot(freq, OTF_aliasing[N//2, :],label = "Aliasing OTF")
plt.plot(freq, OTF_ipsf_difflim[N//2, :]-OTF_aliasing[N//2, :], "--", label="Diffraction-limited OTF\nfrom simulated PSF without aliasing")
plt.vlines([-fc, fc], 0, 1, colors="r", linestyles="--", label = r"$\pm$ cutoff frequency")
plt.title("Diffraction limited and randomly selected OTFs (N°{therandom})", fontsize = BIGGER_SIZE)
plt.xlabel("Angular frequency [cycles/meter]", fontsize=SMALL_SIZE)
plt.ylabel("OTF", fontsize=SMALL_SIZE)
plt.xlim([-fc * 1.2, fc * 1.2])
plt.ylim([0, 1])
plt.xticks(fontsize=SMALL_SIZE*.8)
plt.yticks(fontsize=SMALL_SIZE*.8)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=SMALL_SIZE)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(freq, OTF_truths[12,N//2, :], label="Measured OTF")
plt.plot(freq, OTF_inferences[12,N//2, :]*FREQ_MASK[N//2,:], label="Inference OTF")
plt.plot(freq, OTF_tel[N//2, :], "k", label="Theoretical diffraction-limited OTF")
plt.plot(freq, OTF_ipsf_difflim[N//2, :], label="Diffraction-limited OTF\nfrom simulated PSF with aliasing")
plt.plot(freq, OTF_aliasing[N//2, :],label = "Aliasing OTF")
plt.plot(freq, OTF_ipsf_difflim[N//2, :]-OTF_aliasing[N//2, :], "--", label="Diffraction-limited OTF\nfrom simulated PSF without aliasing")
plt.vlines([-fc, fc], 0, 1, colors="r", linestyles="--", label = r"$\pm$ cutoff frequency")
plt.title("Diffraction limited and randomly selected OTFs", fontsize = BIGGER_SIZE)
plt.xlabel("Angular frequency [cycles/meter]", fontsize=SMALL_SIZE)
plt.ylabel("OTF", fontsize=SMALL_SIZE)
plt.xlim([-fc * 1.2, fc * 1.2])
plt.ylim([0, 1])
plt.xticks(fontsize=SMALL_SIZE*.8)
plt.yticks(fontsize=SMALL_SIZE*.8)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=SMALL_SIZE)
plt.tight_layout()
plt.show()

In [ ]:
OTF_ipsf_difflim = np.abs(myfunctions.otf_from_psf(padded_ipsf_difflim))
OTF_tel = myfunctions.telescope_otf(nu_n_grid)
OTF_aliasing = OTF_ipsf_difflim - OTF_tel
OTF_truths = (np.abs(myfunctions.otf_from_psf(padded_truth))[...,:,:] - OTF_aliasing)*FREQ_MASK
OTF_inferences = (np.abs(myfunctions.otf_from_psf(padded_inferences))[...,:,:] - OTF_aliasing)*FREQ_MASK

In [ ]:
print(np.sum(OTF_inferences[0,:,:]*FREQ_MASK)/ np.sum(OTF_tel*FREQ_MASK))
print(np.sum(OTF_inferences[0,:,:]*FREQ_MASK))
print(np.sum(OTF_tel*FREQ_MASK))

print(np.where(inference_Strehls<1))

inference_Strehls = np.sum(OTF_inferences, axis=(-2, -1)) / np.sum(OTF_tel)
truths_Strehls = np.sum(OTF_truths, axis=(-2, -1)) / np.sum(OTF_tel)
Strehls_difference = truths_Strehls - inference_Strehls
Strehls_rel_fidd = Strehls_difference/truths_Strehls

plt.figure()
plt.plot(inference_Strehls, "r", label = "inf")
plt.plot(truths_Strehls, "k", label = "real")
plt.legend()


## Strehl



In [ ]:

# relative_strehl_dist = (interpolation_strehl - truth_strehl) / truth_strehl

In [ ]:
lower_bound = truth_strehl * 0.9  # -10%
upper_bound = truth_strehl * 1.1  # +10%

plt.close('all')
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={'height_ratios': [3, 2]})

# Top plot: Strehl ratios
ax.plot(truth_strehl, 'k', label=r'$S_{ref}$')
ax.plot(interpolation_strehl, 'r', label=r'$S_{R}$')
ax.set_ylim([0, 1])
ax.fill_between(range(len(truth_strehl)), lower_bound, upper_bound, 
                color='gray', alpha=0.3, edgecolor='none', label='±10% of expected Strehl ratio')
ax.set_ylabel('Strehl ratio [-]', fontsize=MEDIUM_SIZE)
ax.tick_params(axis='y', labelsize=SMALL_SIZE)
ax.legend(fontsize=SMALL_SIZE)
ax.set_title("Comparison of Strehl ratios for instantaneous PSFs", fontsize=BIGGER_SIZE)
ax.set_xlim([0, inference.shape[0]])
ax.grid()

# Bottom plot: Relative distance (ax2 is now a separate subplot below)
ax2.plot(relative_strehl_dist * 100, 'k', label=r"$\frac{S_{R}-S_{ref}}{S_{ref}}$")
ax2.set_ylabel('Relative distance [%]', fontsize=MEDIUM_SIZE, color='k')
ax2.tick_params(axis='y', labelsize=SMALL_SIZE, colors='k')
symetric_lim = max(abs(np.min(relative_strehl_dist) * 110), abs(np.max(relative_strehl_dist) * 110))
ax2.set_ylim([-symetric_lim, symetric_lim])
ax2.legend(fontsize=MEDIUM_SIZE)
ax2.set_xlabel('Iteration [-]', fontsize=MEDIUM_SIZE)
ax2.set_xlim([0, inference.shape[0]])
ax2.grid()
ax2.tick_params(axis='x', labelsize=SMALL_SIZE)

plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')
plt.figure(figsize=(10, 8))
plt.plot(truth_strehl,interpolation_strehl,'.k')
plt.xlim([0,1])
plt.ylim([0,1])
plt.xlabel('Truth Strehl', fontsize=MEDIUM_SIZE)
plt.ylabel('Interpolation Strehl', fontsize=MEDIUM_SIZE)
plt.xticks(fontsize=SMALL_SIZE)
plt.yticks(fontsize=SMALL_SIZE)
plt.title(r'$S_R(S_{ref})$', fontsize=BIGGER_SIZE)
plt.grid(True)
plt.show()

## Rotational average

1. Find center
2. Average the PSF
3. compare